# Part 2 — Step 10: Grad-CAM Visualizations

Generates Grad-CAM heatmaps showing where the fine-tuned EfficientNet-B0 focuses
when classifying DermNet test images as acne vs non-acne.

**Displays 10 predictions (5 acne + 5 non-acne) with class activation maps.**

**Prerequisites:** Run `07_train_classifier.ipynb` (→ `outputs/classifier/best.pth`)
and Section 5 of `part2_colab.ipynb` (→ `outputs/classifier/finetuned.pth`).

In [ ]:
import os
from pathlib import Path

if Path('/content').exists():
    os.chdir('/content/AcneDetection')
print(f'Working directory: {os.getcwd()}')

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from skimage import color as skcolor
from skimage.exposure import match_histograms
from pathlib import Path

%matplotlib inline

DERMNET_DIR = Path('data/dermnet')
PATCH_DIR   = Path('data/patches')
CLF_OUT     = Path('outputs/classifier')
ACNE_FOLDER = 'Acne and Rosacea Photos'
CONF        = 0.5
OUT_FIG     = Path('outputs/figures')
OUT_FIG.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Rebuild Reinhard + histogram matching (must match Section 5 pipeline)
random.seed(42)
acne04_paths = list((PATCH_DIR / 'train' / 'acne').glob('*.jpg'))
sample_paths = random.sample(acne04_paths, min(400, len(acne04_paths)))

pixels = []
for p in sample_paths:
    arr = np.array(Image.open(p).convert('RGB').resize((224, 224))) / 255.0
    pixels.append(skcolor.rgb2lab(arr).reshape(-1, 3))

acne04_pixels   = np.concatenate(pixels, axis=0)
ACNE04_LAB_MEAN = acne04_pixels.mean(axis=0)
ACNE04_LAB_STD  = acne04_pixels.std(axis=0)
ACNE04_REF_IMG  = Image.open(random.choice(acne04_paths)).convert('RGB').resize((224, 224))

def reinhard_normalize(img_pil):
    arr = np.array(img_pil.convert('RGB').resize((224, 224))) / 255.0
    lab = skcolor.rgb2lab(arr)
    src_mean = lab.reshape(-1, 3).mean(axis=0)
    src_std  = lab.reshape(-1, 3).std(axis=0) + 1e-6
    for c in range(3):
        lab[:, :, c] = ((lab[:, :, c] - src_mean[c]) / src_std[c]
                        * ACNE04_LAB_STD[c] + ACNE04_LAB_MEAN[c])
    lab[:, :, 0] = np.clip(lab[:, :, 0], 0, 100)
    lab[:, :, 1] = np.clip(lab[:, :, 1], -128, 127)
    lab[:, :, 2] = np.clip(lab[:, :, 2], -128, 127)
    return Image.fromarray((np.clip(skcolor.lab2rgb(lab), 0, 1) * 255).astype(np.uint8))

def preprocess_dermnet(img_pil):
    img = reinhard_normalize(img_pil)
    arr = np.array(img.convert('RGB').resize((224, 224)))
    matched = match_histograms(arr, np.array(ACNE04_REF_IMG), channel_axis=-1)
    return Image.fromarray(matched.astype(np.uint8))

print('Preprocessing pipeline ready.')

## 1. Load fine-tuned model

In [ ]:
model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.classifier[1].in_features, 2),
)

# Prefer fine-tuned weights from few-shot adaptation; fall back to base checkpoint
ckpt = CLF_OUT / 'finetuned.pth'
if not ckpt.exists():
    ckpt = CLF_OUT / 'best.pth'
    print('finetuned.pth not found — using best.pth')
else:
    print('Loading finetuned.pth')

model.load_state_dict(torch.load(str(ckpt), map_location=device, weights_only=False))
model.to(device).eval()
print('Model ready.')

## 2. Load DermNet test set

In [ ]:
class DermNetBinary(Dataset):
    def __init__(self, split, transform):
        self.samples = []
        root = DERMNET_DIR / split
        for folder in sorted(root.iterdir()):
            label = 1 if folder.name == ACNE_FOLDER else 0
            for img_path in sorted(folder.glob('*')):
                if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    self.samples.append((img_path, label))
        self.transform = transform

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert('RGB')), label

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_ds     = DermNetBinary('test', test_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)

acne_count = sum(1 for _, l in test_ds.samples if l == 1)
print(f'Test set: {len(test_ds)} images  (acne={acne_count}, non-acne={len(test_ds)-acne_count})')

## 3. Grad-CAM on 10 DermNet test images

In [ ]:
random.seed(99)
acne_idx    = [i for i, (_, l) in enumerate(test_ds.samples) if l == 1]
nonacne_idx = [i for i, (_, l) in enumerate(test_ds.samples) if l == 0]
selected    = random.sample(acne_idx, 5) + random.sample(nonacne_idx, 5)

_norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), _norm
])

target_layers = [model.features[-1]]
fig, axes = plt.subplots(2, 5, figsize=(18, 8))

with GradCAM(model=model, target_layers=target_layers) as cam_engine:
    for plot_i, idx in enumerate(selected):
        row, col = divmod(plot_i, 5)
        img_path, true_label = test_ds.samples[idx]

        img_pil = Image.open(img_path).convert('RGB')
        img_pre = preprocess_dermnet(img_pil)   # same pipeline as Section 5

        tensor  = eval_tf(img_pre).unsqueeze(0).to(device)

        with torch.no_grad():
            prob = torch.softmax(model(tensor), dim=1)[0, 1].item()
        pred = 1 if prob >= CONF else 0

        grayscale_cam = cam_engine(
            input_tensor=tensor,
            targets=[ClassifierOutputTarget(pred)]
        )[0]

        raw = np.array(img_pre.resize((224, 224), Image.BILINEAR), dtype=np.float32) / 255.0
        cam_img = show_cam_on_image(raw, grayscale_cam, use_rgb=True)

        true_str = 'acne' if true_label == 1 else 'non-acne'
        pred_str = 'acne' if pred == 1 else 'non-acne'
        color    = 'green' if true_label == pred else 'red'

        axes[row, col].imshow(cam_img)
        axes[row, col].set_title(
            f'True: {true_str}\nPred: {pred_str} ({prob:.2f})',
            fontsize=8, color=color
        )
        axes[row, col].axis('off')

plt.suptitle(
    'Grad-CAM — Fine-tuned EfficientNet-B0 on DermNet Test\n'
    'Row 1: acne samples  |  Row 2: non-acne samples  (green=correct, red=wrong)',
    fontsize=12
)
plt.tight_layout()
plt.savefig(OUT_FIG / 'gradcam_dermnet.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/gradcam_dermnet.png')

## 4. Reflection

**Domain transfer performance**: The model trained purely on ACNE04 (tight 224×224 skin patches) scored AUROC < 0.5 on DermNet — below random chance — because DermNet images are full clinical photos with backgrounds, watermarks, and varied compositions absent from training.

**Mitigation strategies**:
1. **Augmentation** (ColorJitter, RandomResizedCrop, GaussianBlur) during training made the model robust to lighting and scale differences between domains.
2. **Few-shot fine-tuning**: 10 acne + 10 non-acne DermNet training samples at lr=1e-5 for 30 epochs allowed the model to partially adapt to the clinical distribution.

**Grad-CAM insights**: On DermNet images where the model predicts acne, the heatmap highlights central skin regions, confirming the model is attending to skin texture rather than backgrounds. On misclassified images, attention often lands on irrelevant areas (clothing, watermarks), illustrating the domain gap.

**Remaining challenges**: DermNet's acne folder contains mislabeled and watermarked images, and the 92% class imbalance makes F1 and AUROC far more informative than accuracy. Future work: center-crop preprocessing, histogram matching to ACNE04 statistics, or a full CycleGAN-based style transfer.